# What changes when one Hurricane Lee forecast is withdrawn?

Thirty-one NOAA GEFS members forecast maximum sustained wind at the same target time. Score the ensemble against the later NHC best-track estimate, drop a member, and score the surviving 30. Change the cycle or member ID below and run the cells again.

The last cell runs RecastCRPS's published WebAssembly module in the Python (Pyodide) kernel. A desktop Python kernel uses the RecastCRPS extension if installed. The notebook downloads a 124-row public CSV and, in the browser, a 37 KB module. Forecast initialization times describe archived cycles. The verification wind was assigned after the storm.

[Source and transformations](https://github.com/pgg898200-ship-it/streamsieve/blob/a71e449d6746caceb733d0e40b888f2c9fb443f5/integrations/recastcrps/replay/README.md) · [NOAA/NHC Hurricane Lee report](https://www.nhc.noaa.gov/data/tcr/AL132023_Lee.pdf)

In [1]:
import hashlib
import sys

async def public_bytes(url):
    if sys.platform == "emscripten":
        from pyodide.http import pyfetch
        response = await pyfetch(url)
        if not response.ok:
            raise RuntimeError(f"HTTP {response.status}: {url}")
        return await response.bytes()
    from urllib.request import urlopen
    with urlopen(url, timeout=30) as response:
        return response.read()

import csv
import io

base = "https://raw.githubusercontent.com/pgg898200-ship-it/streamsieve/a71e449d6746caceb733d0e40b888f2c9fb443f5/integrations/recastcrps"
data = await public_bytes(f"{base}/replay/lee_gefs.csv")
assert hashlib.sha256(data).hexdigest() == "3314ea7d0e64f5b38ee5d92be288c886ea9bcc4aec99ff0d88712979c5b8ce2a"
cycles = list(csv.DictReader(io.StringIO(data.decode("utf-8"))))
print(f"{len(cycles)} archived cycles across {len(set(row['target_utc'] for row in cycles))} target times")

124 archived cycles across 11 target times


## Choose the forecast and member

The default is the control member from the 06 UTC initialization on September 8, 2023. The target time is 12 UTC that day. Change `withdraw` to any of `AP01` through `AP30` to inspect a perturbed member.

In [2]:
target_utc = "2023090812"
init_utc = "2023090806"
withdraw = "AC00"

matches = [row for row in cycles if row["target_utc"] == target_utc and row["init_utc"] == init_utc]
if not matches:
    raise ValueError("Choose a target_utc and init_utc present in the CSV.")
row = matches[0]
members = {key: int(value) for key, value in row.items() if key.startswith(("AC", "AP"))}
observed = int(row["observed_kt"])
print(f"{withdraw} forecast {members[withdraw]} kt. The verifying intensity was {observed} kt.")

AC00 forecast 63 kt. The verifying intensity was 135 kt.


## Calculate the change directly

For equal weights the empirical continuous ranked probability score is the mean absolute error minus half the mean pairwise distance over all ordered pairs. Keep the calculation as a fraction so the final decimal is the only rounding step. Lower scores indicate a closer match to this observation.

In [3]:
from fractions import Fraction

def empirical_crps(values, outcome):
    n = len(values)
    errors = sum(abs(value - outcome) for value in values)
    spread = sum(abs(a - b) for i, a in enumerate(values) for b in values[i + 1:])
    return Fraction(errors, n) - Fraction(spread, n * n)

full = empirical_crps(list(members.values()), observed)
reduced = empirical_crps([v for k, v in members.items() if k != withdraw], observed)
print(f"All 31 members:   {float(full):.8f} kt")
print(f"Without {withdraw}:     {float(reduced):.8f} kt")
print(f"Score change:     {float(reduced - full):+.8f} kt")

All 31 members:   69.46097815 kt
Without AC00:     69.40666667 kt
Score change:     -0.05431148 kt


## Check the release against the direct calculation

The browser build exposes the same stable-slot operations used to update a live ensemble. Set each member once, withdraw the selected slot and check both answers. A single target illustrates the arithmetic. Decisions about removing a member from an operational forecast require data across many targets.

In [4]:
import math

if sys.platform == "emscripten":
    from js import Object, Uint8Array, WebAssembly
    from pyodide.ffi import to_js

    wasm = await public_bytes(f"{base}/demo/recast.wasm")
    assert hashlib.sha256(wasm).hexdigest() == "3714af22d19d99efd12ab7a86eedb679f2e15e71b825abb2b32f2ab10fa81cda"
    compiled = await WebAssembly.instantiate(Uint8Array.new(to_js(wasm)), Object.new())
    kernel = compiled.instance.exports
    kernel._initialize()
    assert kernel.reset_ensemble(255, len(members)) == 0
    for slot, wind in enumerate(members.values()):
        assert kernel.set_member(slot, wind, 1) == 0
    native_full = kernel.ensemble_score(observed)
    assert kernel.erase_member(list(members).index(withdraw)) == 0
    native_reduced = kernel.ensemble_score(observed)
else:
    from recastcrps import Ensemble  # See the source installation below for desktop kernels.
    panel = Ensemble(range(256), slots=len(members))
    for slot, wind in enumerate(members.values()):
        panel.set(slot, wind)
    native_full = panel.score(observed)
    panel.erase(list(members).index(withdraw))
    native_reduced = panel.score(observed)

assert math.isclose(native_full, float(full), abs_tol=1e-12, rel_tol=0)
assert math.isclose(native_reduced, float(reduced), abs_tol=1e-12, rel_tol=0)
print("Both release scores match the direct calculation.")

Both release scores match the direct calculation.


[Install RecastCRPS for desktop Python](https://github.com/pgg898200-ship-it/streamsieve/tree/main/integrations/recastcrps#try-it) · [Explore the Hurricane Lee replay](https://huggingface.co/spaces/pgg-engineering/recastcrps)